## Scraping Respiratory Virus Detections Data

I will now scrape the respiratory virus detections data from the Canadian public health website. The goal is to extract 'Table 2' and convert it into a pandas DataFrame.

<a href="https://colab.research.google.com/github/techseeko/AAI614_Haidar/blob/main/Week-3/Saoud_VirusDetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import requests # Keeping import for now, but not used in new logic
from bs4 import BeautifulSoup
import pandas as pd
import asyncio
from playwright.async_api import async_playwright

In [10]:
# Install Playwright Python package
print("Installing playwright Python package...")
!pip install playwright > /dev/null

Installing playwright Python package...


In [11]:
# --- Playwright setup and scraping logic ---

# Install Playwright system dependencies and browser binaries directly in this cell
# This ensures that Playwright's browser can launch correctly.
print("Updating apt-get and installing essential system dependencies...")
!apt-get update > /dev/null
!apt-get install -y libnss3 libxss1 libatk1.0-0 libgtk-3-0 libgdk-pixbuf2.0-0 libxcomposite1 libgbm-dev libcups2 > /dev/null # Essential common dependencies
print("Installing additional Playwright system dependencies...")
!playwright install-deps # Let this output be visible in case of errors
print("Installing Playwright browser binaries...")
!playwright install > /dev/null # Suppress verbose output

Updating apt-get and installing essential system dependencies...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Installing additional Playwright system dependencies...
Installing dependencies...
Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'mai

In [12]:
url = 'https://www.canada.ca/en/public-health/services/surveillance/respiratory-virus-detections-canada/2018-2019/respiratory-virus-detections-isolations-week-01-ending-january-5-2019.html'

async def get_page_content(url_to_scrape):
    async with async_playwright() as p:
        browser = await p.chromium.launch()
        page = await browser.new_page()
        try:
            # Increased timeout for page load in Playwright to 90 seconds
            await page.goto(url_to_scrape, timeout=90000)
            content = await page.content()
            return content
        except Exception as e:
            print(f"Playwright failed to load page: {e}")
            return None
        finally:
            await browser.close()

# Use top-level await for Colab environment
try:
    page_content = await get_page_content(url)

    if page_content:
        print("Successfully fetched page content using Playwright.")
        soup = BeautifulSoup(page_content, 'html.parser')

        # Find 'Table 2'. Based on inspection, it might be the second table on the page.
        # More robust way: find the heading 'Table 2' and then find the next table sibling.
        table_heading = soup.find('h3', string=lambda text: text and 'Table 2' in text)
        if table_heading:
            table = table_heading.find_next_sibling('table')
        else:
            # Fallback if the heading isn't found, try to get the second table
            tables = soup.find_all('table')
            if len(tables) > 1:
                table = tables[1] # Assuming 'Table 2' is the second table
            else:
                table = None
                print("Could not find 'Table 2' on the page.")

        if table:
            headers = [th.text.strip() for th in table.find('thead').find_all('th')]
            data = []
            for row in table.find('tbody').find_all('tr'):
                cells = row.find_all('td')
                # Exclude rows that are just headers or empty
                if cells:
                    data.append([cell.text.strip() for cell in cells])

            df = pd.DataFrame(data, columns=headers)
            print("DataFrame created successfully from Table 2 using Playwright.")
            display(df.head())
            df.info()
        else:
            print("No table found for extraction after Playwright fetch.")
    else:
        print("Failed to retrieve page content using Playwright.")

except Exception as e:
    print(f"An error occurred during Playwright scraping: {e}")

Successfully fetched page content using Playwright.
DataFrame created successfully from Table 2 using Playwright.


,Reporting Laboratory,Flu Tested,A(H1N1)pdm09 Positive,A(H3) Positive,A(UnS) Positive,Total Flu A Positive,Total Flu B Positive,RSV Tested,RSV Positive,PIV Tested,...,PIV 4 Positive,Other PIV Positive,Adeno Tested,Adeno Positive,hMPV Tested,hMPV Positive,Entero/Rhino Tested,Entero/Rhino Positive,Coron Tested,Coron Positive
0,Newfoundland,1299,1,0,113,114,1,1299,91,1299,...,0,0,1299,12,1299,8,1299,200,N.A.,N.A.
1,Prince Edward Island,307,38,0,0,38,0,305,5,53,...,3,0,48,5,48,0,48,21,48,0
2,Nova Scotia,864,0,0,52,52,1,869,45,322,...,1,0,322,0,322,3,322,53,322,1
3,New Brunswick,4271,42,1,715,758,2,4274,131,1185,...,29,0,1185,84,1185,7,1185,201,1185,6
4,Atlantic,6741,81,1,880,962,4,6747,272,2859,...,33,0,2854,101,2854,18,2854,475,1555,7


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41 entries, 0 to 40
Data columns (total 23 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Reporting Laboratory   41 non-null     object
 1   Flu Tested             41 non-null     object
 2   A(H1N1)pdm09 Positive  41 non-null     object
 3   A(H3) Positive         41 non-null     object
 4   A(UnS) Positive        41 non-null     object
 5   Total Flu A Positive   41 non-null     object
 6   Total Flu B Positive   41 non-null     object
 7   RSV Tested             41 non-null     object
 8   RSV Positive           41 non-null     object
 9   PIV Tested             41 non-null     object
 10  PIV 1 Positive         41 non-null     object
 11  PIV 2 Positive         41 non-null     object
 12  PIV 3 Positive         41 non-null     object
 13  PIV 4 Positive         41 non-null     object
 14  Other PIV Positive     41 non-null     object
 15  Adeno Tested           41